# 06 — Nucleotide Transformer v2 with SeqTrainer

This notebook demonstrates how to use **SeqTrainer APIs** to fine-tune a Nucleotide Transformer v2 backbone for two promoter tasks:

1. Binary classification (`high_activity`)
2. Regression (`target`)


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, mean_squared_error, r2_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

from seqtrainer.data.materialized import MaterializedDataset
from seqtrainer.torch import get_nucleotide_transformer_v2_backbone


In [ ]:
# Load promoter sequences from the repository example dataset.
df = pd.read_csv("../../data/dataset_builder/unprocessed_dataset.csv")
df.head()


In [ ]:
# Create SeqTrainer materialized dataset so we stay on package interfaces.
examples = [
    {"sequence": row.sequence, "target": float(row.target)}
    for row in df.itertuples(index=False)
]
dataset = MaterializedDataset(examples, metadata={"source": "dataset_builder/unprocessed_dataset.csv"})

train_ds, val_ds, test_ds = dataset.train_val_test_split(train_size=0.7, val_size=0.15, test_size=0.15, seed=7)
len(train_ds.examples), len(val_ds.examples), len(test_ds.examples)


In [ ]:
# Resolve the Nucleotide Transformer v2 backbone via SeqTrainer helpers.
backbone = get_nucleotide_transformer_v2_backbone()
backbone


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(backbone.config["tokenizer_name"], trust_remote_code=True)

def build_hf_dataset(rows: list[dict], *, task: str, threshold: float | None = None) -> Dataset:
    out = []
    for item in rows:
        target = float(item["target"])
        if task == "classification":
            label = int(target >= float(threshold))
        elif task == "regression":
            label = target
        else:
            raise ValueError(f"Unsupported task: {task}")
        out.append({"sequence": item["sequence"], "labels": label})
    return Dataset.from_list(out)

def tokenize_batch(batch):
    return tokenizer(batch["sequence"], truncation=True, max_length=backbone.config["max_length"])


## 1) Promoter activity classification

We binarize with the **median of train targets** to avoid leakage.

In [ ]:
threshold = float(np.median([x["target"] for x in train_ds.examples]))

train_cls = build_hf_dataset(train_ds.examples, task="classification", threshold=threshold).map(tokenize_batch, batched=True)
val_cls = build_hf_dataset(val_ds.examples, task="classification", threshold=threshold).map(tokenize_batch, batched=True)
test_cls = build_hf_dataset(test_ds.examples, task="classification", threshold=threshold).map(tokenize_batch, batched=True)

model_cls = AutoModelForSequenceClassification.from_pretrained(
    backbone.config["model_name"],
    num_labels=2,
    trust_remote_code=True,
)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

def classification_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds),
    }

args_cls = TrainingArguments(
    output_dir="../../outputs/ntv2_promoter_classification",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    report_to="none",
)

trainer_cls = Trainer(
    model=model_cls,
    args=args_cls,
    train_dataset=train_cls,
    eval_dataset=val_cls,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=classification_metrics,
)

# trainer_cls.train()
# cls_test_metrics = trainer_cls.evaluate(test_cls)
# cls_test_metrics


## 2) Promoter activity regression

In [ ]:
train_reg = build_hf_dataset(train_ds.examples, task="regression").map(tokenize_batch, batched=True)
val_reg = build_hf_dataset(val_ds.examples, task="regression").map(tokenize_batch, batched=True)
test_reg = build_hf_dataset(test_ds.examples, task="regression").map(tokenize_batch, batched=True)

model_reg = AutoModelForSequenceClassification.from_pretrained(
    backbone.config["model_name"],
    num_labels=1,
    problem_type="regression",
    trust_remote_code=True,
)

def regression_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.reshape(-1)
    labels = labels.reshape(-1)
    rmse = float(np.sqrt(mean_squared_error(labels, preds)))
    return {"rmse": rmse, "r2": r2_score(labels, preds)}

args_reg = TrainingArguments(
    output_dir="../../outputs/ntv2_promoter_regression",
    learning_rate=2e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    report_to="none",
)

trainer_reg = Trainer(
    model=model_reg,
    args=args_reg,
    train_dataset=train_reg,
    eval_dataset=val_reg,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=regression_metrics,
)

# trainer_reg.train()
# reg_test_metrics = trainer_reg.evaluate(test_reg)
# reg_test_metrics


## Notes

- Remove comments around `train()`/`evaluate()` cells to run end-to-end training.
- Nucleotide Transformer v2 is large; use a GPU runtime and adjust batch size as needed.
- You can swap a custom checkpoint by editing the `BackboneSpec` config returned by SeqTrainer.
